# Лабораторная работа №9: Оценка качества RAG-систем

## Цель работы
Изучить методы и метрики для автоматической оценки качества RAG (Retrieval-Augmented Generation) систем, включая оценку релевантности retrieved документов, точности ответов и полноты информации.

## Теоретическая часть

### Зачем нужна оценка RAG?
RAG-системы состоят из двух компонентов:
1. **Retriever** — поиск релевантных документов
2. **Generator** — генерация ответа на основе найденных документов

Ошибки могут возникать на каждом этапе:
- Нерелевантные документы найдены
- Важная информация упущена
- Ответ содержит галлюцинации
- Ответ не соответствует вопросу

### Ключевые метрики оценки

#### 1. Метрики Retrieval
- **Precision@K** — доля релевантных документов среди топ-K
- **Recall@K** — доля найденных релевантных документов от всех релевантных
- **MRR (Mean Reciprocal Rank)** — средний обратный ранг первого релевантного документа
- **NDCG (Normalized Discounted Cumulative Gain)** — учет позиции релевантных документов

#### 2. Метрики Generation
- **Faithfulness (Верность)** — насколько ответ основан на контексте
- **Answer Relevance (Релевантность ответа)** — насколько ответ соответствует вопросу
- **Context Precision** — точность предоставленного контекста
- **Context Recall** — полнота информации в контексте

### Фреймворки для оценки
- **RAGAS** — комплексная оценка RAG-систем
- **DeepEval** — фреймворк для оценки LLM
- **TruLens** — трассировка и оценка LLM-приложений
- **LangSmith** — платформа для отладки и мониторинга

## Практическая часть

### Установка зависимостей

In [ ]:
!pip install ragas langchain langchain-openai datasets -q

### Импорт библиотек и настройка

In [ ]:
import os
from google.colab import userdata

# Настройка API ключа
try:
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except:
    os.environ["OPENAI_API_KEY"] = "your-api-key-here"

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
    answer_similarity,
    answer_correctness
)
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

### Создание тестового датасета

Для оценки RAG нужен датасет с:
- question — вопрос
- answer — сгенерированный ответ
- contexts — контексты, использованные для генерации
- ground_truth — правильный ответ (опционально)

In [ ]:
# Пример данных для оценки
data_samples = {
    'question': [
        'Какие основные преимущества использования RAG?',
        'Как работает механизм внимания в трансформерах?',
        'Что такое квантование моделей и зачем оно нужно?',
        'В чем разница между fine-tuning и prompt engineering?',
        'Какие существуют методы оптимизации инференса LLM?'
    ],
    'answer': [
        'Основные преимущества RAG включают: возможность использования актуальных данных, снижение галлюцинаций модели, прозрачность источников информации, экономия ресурсов на дообучение.',
        'Механизм внимания вычисляет веса для каждого токена в последовательности, позволяя модели фокусироваться на наиболее релевантных частях входных данных при генерации каждого выходного токена.',
        'Квантование — это уменьшение точности чисел в весах модели (например, с float32 до int8), что снижает требования к памяти и ускоряет инференс с минимальной потерей качества.',
        'Fine-tuning изменяет веса модели на обучающих данных, тогда как prompt engineering подбирает оптимальные формулировки запросов без изменения весов модели.',
        'Методы оптимизации включают: квантование, дистилляцию, pruning (прореживание), использование специализированных движков (vLLM, TensorRT), батчинг запросов.'
    ],
    'contexts': [
        ['RAG позволяет использовать внешние источники знаний без дообучения модели. Это снижает галлюцинации и обеспечивает актуальность информации.'],
        ['Attention mechanism computes attention scores for each token pair. Self-attention allows the model to weigh the importance of different words in a sequence.'],
        ['Quantization reduces model size by representing weights with lower precision numbers. INT8 and INT4 are common quantization formats.'],
        ['Fine-tuning updates model weights on domain-specific data. Prompt engineering crafts better inputs without modifying the model.'],
        ['Inference optimization techniques include quantization, distillation, pruning, and using specialized inference engines like vLLM and TensorRT.']
    ],
    'ground_truth': [
        'RAG обеспечивает доступ к актуальным данным, снижает галлюцинации, повышает прозрачность и экономит ресурсы на обучение.',
        'Механизм внимания вычисляет веса значимости между токенами, позволяя модели фокусироваться на релевантной информации.',
        'Квантование уменьшает точность представления весов модели для снижения потребления памяти и ускорения работы.',
        'Fine-tuning изменяет параметры модели, а prompt engineering только меняет входные данные.',
        'Оптимизация инференса достигается через квантование, дистилляцию, прореживание и специализированные движки.'
    ]
}

# Создание Dataset для ragas
dataset = Dataset.from_dict(data_samples)
print(f"Создан датасет из {len(dataset)} примеров")
print(f"\nПример вопроса: {data_samples['question'][0]}")
print(f"Пример ответа: {data_samples['answer'][0][:100]}...")

### Оценка с помощью RAGAS

In [ ]:
# Инициализация LLM для оценки
llm = ChatOpenAI(model="gpt-3.5-turbo")

# Выбор метрик для оценки
metrics = [
    faithfulness,           # Верность ответа контексту
    answer_relevancy,       # Релевантность ответа вопросу
    context_precision,      # Точность контекста
    context_recall,         # Полнота контекста
    answer_similarity,      # Сходство с ground truth
    answer_correctness      # Общая корректность
]

print("Начало оценки RAG-системы...")
print("Это может занять несколько минут...\n")

# Запуск оценки
result = evaluate(
    dataset,
    metrics=metrics,
    llm=llm,
    raise_exceptions=False
)

print("Оценка завершена!")
print("\n" + "="*60)
print("РЕЗУЛЬТАТЫ ОЦЕНКИ")
print("="*60)

In [ ]:
# Вывод результатов
import pandas as pd

# Преобразование результатов в DataFrame
df_result = result.to_pandas()
print(df_result)

# Средние значения по метрикам
print("\n" + "="*60)
print("СРЕДНИЕ ЗНАЧЕНИЯ МЕТРИК")
print("="*60)

for metric in metrics:
    metric_name = metric.__name__ if hasattr(metric, '__name__') else str(metric)
    avg_score = df_result[f'{metric_name}'].mean()
    print(f"{metric_name:25s}: {avg_score:.3f}")

### Детальный анализ отдельных примеров

In [ ]:
# Анализ конкретного примера
example_idx = 0

print(f"\n{'='*60}")
print(f"АНАЛИЗ ПРИМЕРА #{example_idx + 1}")
print(f"{'='*60}\n")

print(f"Вопрос: {data_samples['question'][example_idx]}\n")
print(f"Ground Truth: {data_samples['ground_truth'][example_idx]}\n")
print(f"Ответ модели: {data_samples['answer'][example_idx]}\n")
print(f"Контекст: {data_samples['contexts'][example_idx][0]}\n")

print("Оценки по метрикам:")
for metric in metrics:
    metric_name = metric.__name__ if hasattr(metric, '__name__') else str(metric)
    score = df_result[f'{metric_name}'].iloc[example_idx]
    print(f"  {metric_name:25s}: {score:.3f}")

### Создание простой RAG-системы для тестирования

In [ ]:
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.chains import RetrievalQA

# Документы для базы знаний
documents_text = """
RAG (Retrieval-Augmented Generation) — это архитектура, сочетающая поиск информации с генерацией текста.
Основные компоненты RAG: retriever (поисковый модуль) и generator (языковая модель).
Преимущества RAG: использование актуальных данных, снижение галлюцинаций, прозрачность источников.

Трансформеры используют механизм self-attention для обработки последовательностей.
Attention вычисляет веса значимости между всеми парами токенов в последовательности.
Multi-head attention позволяет модели одновременно фокусироваться на разных аспектах входных данных.

Квантование моделей уменьшает размер модели за счет снижения точности весов.
INT8 квантование сокращает память в 4 раза по сравнению с float32.
QLoRA комбинирует квантование с эффективным fine-tuning.

Fine-tuning обновляет веса предобученной модели на специфичных данных.
Prompt engineering создает эффективные запросы без изменения модели.
PEFT (Parameter-Efficient Fine-Tuning) методы включают LoRA, adapter layers.

Оптимизация инференса включает квантование, дистилляцию, pruning.
vLLM использует PagedAttention для эффективного управления памятью.
Батчинг запросов повышает пропускную способность системы.
"""

# Разбиение на документы
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=50)
texts = text_splitter.split_text(documents_text)
docs = [Document(page_content=t) for t in texts]

# Создание векторного хранилища
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(docs, embeddings)

# Создание retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Создание QA цепочки
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-3.5-turbo"),
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

print("RAG-система создана!")

In [ ]:
# Тестирование RAG-системы
test_questions = [
    "Что такое RAG и какие у него преимущества?",
    "Как работает механизм attention?",
    "Что такое квантование моделей?"
]

for question in test_questions:
    print(f"\n{'='*60}")
    print(f"Вопрос: {question}")
    print(f"{'='*60}")
    
    result = qa_chain({"query": question})
    
    print(f"Ответ: {result['result']}")
    print(f"\nИсточники:")
    for i, doc in enumerate(result['source_documents'], 1):
        print(f"  {i}. {doc.page_content[:150]}...")

### Задание 1: Сбор собственного датасета для оценки

Создайте датасет из 10-15 вопросов и ответов для вашей предметной области:
1. Сформулируйте вопросы
2. Подготовьте ground truth ответы
3. Соберите релевантные контексты
4. Запустите вашу RAG-систему для получения ответов
5. Оцените качество с помощью RAGAS

In [ ]:
# Ваш код здесь
# Создайте собственный датасет и оцените RAG-систему

### Задание 2: Сравнение различных конфигураций RAG

Сравните качество при различных настройках:
- Разное количество retrieved документов (k=1, 3, 5)
- Разные модели эмбеддингов
- Разные промпты для генерации
- С фильтром по метаданным и без

In [ ]:
# Ваш код здесь
# Сравните различные конфигурации RAG

### Задание 3: Анализ ошибок

Проанализируйте примеры с низкими оценками:
- Почему система дала плохой ответ?
- Проблема в retrieval или generation?
- Какие улучшения можно внести?

In [ ]:
# Ваш код здесь
# Проанализируйте ошибки и предложите улучшения

## Контрольные вопросы

1. Какие метрики RAGAS наиболее важны для оценки качества RAG-системы и почему?
2. В чем разница между faithfulness и answer relevancy?
3. Как интерпретировать значения метрик (что считать хорошим результатом)?
4. Какие проблемы могут возникнуть при автоматической оценке и как их минимизировать?
5. Как собрать репрезентативный тестовый датасет для оценки?

## Требования к отчету

1. Описание созданного тестового датасета
2. Таблица с результатами оценки по всем метрикам
3. Сравнительный анализ различных конфигураций RAG
4. Анализ ошибок и предложения по улучшению
5. Ответы на контрольные вопросы
6. Выводы о качестве реализованной RAG-системы